# 5 — Simple RAG evaluation

This notebook illustrates a minimal evaluation workflow:

1. prepare questions with known relevant source IDs and reference answers;
2. run the RAG pipeline on these questions;
3. evaluate retrieval and generated answers.

> The data below are fictitious examples. Replace them with questions and source IDs validated from the maintenance logs.

In [ ]:
evaluation_set = [
    {
        "id": "q1",
        "question": "What action was taken for the faulty saloon light?",
        "relevant_ids": ["doc-12"],
        "reference_answer": "The faulty lamp was replaced.",
    },
    {
        "id": "q2",
        "question": "Which actions were used for the documented air dryer faults?",
        "relevant_ids": ["doc-25", "doc-31"],
        "reference_answer": "The air dryer was inspected and the faulty valve was replaced.",
    },
]

`rag_results` represents the output of the RAG pipeline. For each question, save the ranked source IDs returned by the retriever and the generated answer.

In [ ]:
rag_results = {
    "q1": {
        "retrieved_ids": ["doc-12", "doc-8", "doc-40"],
        "answer": "The faulty lamp was replaced.",
    },
    "q2": {
        "retrieved_ids": ["doc-9", "doc-25", "doc-31"],
        "answer": "The logs mention an inspection and the replacement of a faulty valve.",
    },
}

## Retrieval metrics

- **Hit@K:** at least one relevant document appears in the first K results.
- **Recall@K:** proportion of relevant documents found in the first K results.
- **MRR@K:** rewards systems that rank the first relevant document near the top.

In [ ]:
from statistics import mean

def hit_at_k(retrieved, relevant, k):
    return int(bool(set(retrieved[:k]) & set(relevant)))

def recall_at_k(retrieved, relevant, k):
    return len(set(retrieved[:k]) & set(relevant)) / len(set(relevant))

def reciprocal_rank_at_k(retrieved, relevant, k):
    for rank, source_id in enumerate(retrieved[:k], start=1):
        if source_id in relevant:
            return 1 / rank
    return 0

k = 3
scores = []

for item in evaluation_set:
    retrieved = rag_results[item["id"]]["retrieved_ids"]
    relevant = item["relevant_ids"]
    scores.append({
        "hit": hit_at_k(retrieved, relevant, k),
        "recall": recall_at_k(retrieved, relevant, k),
        "rr": reciprocal_rank_at_k(retrieved, relevant, k),
    })

print(f"Hit@{k}:   {mean(score['hit'] for score in scores):.2f}")
print(f"Recall@{k}: {mean(score['recall'] for score in scores):.2f}")
print(f"MRR@{k}:    {mean(score['rr'] for score in scores):.2f}")

## Answer similarity

The cosine similarity between sentence embeddings gives a simple automatic comparison between generated and reference answers. A high score does not guarantee that the answer is correct or supported by the retrieved documents.

In [ ]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")
references = [item["reference_answer"] for item in evaluation_set]
answers = [rag_results[item["id"]]["answer"] for item in evaluation_set]

reference_embeddings = model.encode(references, normalize_embeddings=True)
answer_embeddings = model.encode(answers, normalize_embeddings=True)
similarities = (reference_embeddings * answer_embeddings).sum(axis=1)

for item, similarity in zip(evaluation_set, similarities):
    print(f"{item['id']}: {similarity:.2f}")
print(f"Mean similarity: {similarities.mean():.2f}")

## Manual check

Automatic metrics are not sufficient. For a sample of questions, read the retrieved documents and check whether the generated answer is correct, complete, and supported by those documents. Use the same evaluation set when comparing two configurations.